In [1]:
import os
import warnings

import pandas as pd

# Path to result data (relative to notebooks/ directory)
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("")), "result", "stat")

# Display names for methods
METHOD_DISPLAY = {
    "cls": "AS", "thd": "DT", "lrn": "LR", "ste": "STE",
    "exact": "EX", "rel": "RR", "root": "N1",
}

# Baseline methods (no param in filename)
BASELINE_METHODS = {"exact", "rel", "root"}

In [2]:
def load_results(problem, sizes, methods, penalty):
    """Load experiment results and reshape into a comparison table."""
    results = []
    for size in sizes:
        for method in methods:
            results.append(retrieve_data(problem, size, method, penalty))
    df = pd.DataFrame(results)
    return reshape_dataframe(df, problem, sizes, methods)


def retrieve_data(problem, size, method, penalty):
    """Retrieve metrics from a single experiment CSV file."""
    # Parse projection flag
    projection = method.endswith("-p")
    base_method = method[:-2] if projection else method
    display_name = METHOD_DISPLAY[base_method] + ("-P" if projection else "")

    # Build file path: baseline methods have no penalty
    size_str = str(size) if problem == "rb" else f"{size}-{size}"
    suffix = "-p" if projection else ""
    if base_method in BASELINE_METHODS:
        filename = f"{problem}_{base_method}_{size_str}{suffix}.csv"
    else:
        filename = f"{problem}_{base_method}{penalty}_{size_str}{suffix}.csv"
    csv_path = os.path.join(DATA_DIR, filename)

    # Default metrics for missing files
    metrics = {
        "Method": display_name,
        "Problem Size": size,
        "Obj Mean": None,
        "Obj Median": None,
        "% Infeasible": None,
        "% Unsolved": None,
        "Time (Sec)": None,
    }

    if not os.path.exists(csv_path):
        warnings.warn(f"File not found: {csv_path}")
        return metrics

    df = pd.read_csv(csv_path)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        metrics.update({
            "Obj Mean": df["Obj Val"].mean(),
            "Obj Median": df["Obj Val"].median(),
            "% Infeasible": (
                (df["Num Violations"] > 0).mean() * 100
                if "Num Violations" in df.columns
                else (df["Constraints Viol"] > 0).mean() * 100
            ),
            "% Unsolved": df["Obj Val"].isna().mean() * 100,
            "Time (Sec)": df["Elapsed Time"].mean(),
        })
    return metrics


def reshape_dataframe(df, problem, sizes, methods):
    """Reshape results from long format into a method x metric comparison table."""
    display_methods = [
        METHOD_DISPLAY[m[:-2]] + "-P" if m.endswith("-p") else METHOD_DISPLAY[m]
        for m in methods
    ]
    metric_names = ["Obj Mean", "Obj Median", "% Infeasible", "% Unsolved", "Time (Sec)"]

    rows = []
    for method in display_methods:
        for metric in metric_names:
            row = {"Method": method, "Metric": metric}
            for s in sizes:
                values = df[(df["Method"] == method) & (df["Problem Size"] == s)][metric].values
                if len(values) != 1:
                    raise ValueError(
                        f"Expected 1 value for {method}/{metric}/size={s}, got {len(values)}"
                    )
                col = f"{s * 2}x4" if problem == "rb" else f"{s}x{s}"
                row[col] = values[0]
            rows.append(row)
    return pd.DataFrame(rows)

### Convex Quadratic (QP)

In [3]:
sizes = [20, 50, 100, 200, 500, 1000]
methods = ["cls", "cls-p", "thd", "thd-p", "exact", "rel", "root", "lrn", "ste"]
load_results(problem="cq", sizes=sizes, methods=methods, penalty=1.0)

,Method,Metric,20x20,50x50,100x100,200x200,500x500,1000x1000
0,AS,Obj Mean,-4.879978e+00,-1.512641e+01,-1.945163e+01,-41.032806,-92.320995,-183.250237
1,AS,Obj Median,-4.917381e+00,-1.516119e+01,-1.943988e+01,-41.034512,-92.308155,-183.284159
2,AS,% Infeasible,7.800000e+01,9.000000e+01,9.800000e+01,98.000000,100.000000,100.000000
3,AS,% Unsolved,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
4,AS,Time (Sec),1.804590e-05,1.789808e-05,1.879692e-05,0.000038,0.000022,0.000022
5,AS-P,Obj Mean,-4.819501e+00,-1.508453e+01,-1.933796e+01,-40.975396,-92.144262,-182.959230
6,AS-P,Obj Median,-4.829980e+00,-1.506975e+01,-1.934205e+01,-40.979979,-92.168870,-182.995273
7,AS-P,% Infeasible,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
8,AS-P,% Unsolved,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
9,AS-P,Time (Sec),6.433816e-03,3.535104e-03,1.582677e-03,0.001099,0.000826,0.000551


### Simple Non-Convex

In [4]:
sizes = [20, 50, 100, 200, 500, 1000]
methods = ["cls", "cls-p", "thd", "thd-p", "exact", "rel", "root", "lrn", "ste"]
load_results(problem="nc", sizes=sizes, methods=methods, penalty=1.0)

,Method,Metric,20x20,50x50,100x100,200x200,500x500,1000x1000
0,AS,Obj Mean,-0.248311,-1.200589e+00,-2.036498,-3.578413,-4.804356,-18.082888
1,AS,Obj Median,-0.236336,-1.217822e+00,-2.040272,-3.652391,-10.075942,-18.085712
2,AS,% Infeasible,27.000000,8.100000e+01,72.000000,90.000000,93.000000,99.000000
3,AS,% Unsolved,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
4,AS,Time (Sec),0.000020,2.062559e-05,0.000020,0.000035,0.000023,0.000026
5,AS-P,Obj Mean,-0.245407,-1.186733e+00,-2.024602,-3.575743,-4.914447,-18.032323
6,AS-P,Obj Median,-0.233918,-1.201151e+00,-2.032327,-3.658996,-10.038494,-18.075630
7,AS-P,% Infeasible,0.000000,0.000000e+00,0.000000,0.000000,1.000000,0.000000
8,AS-P,% Unsolved,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
9,AS-P,Time (Sec),0.004982,5.581918e-03,0.001117,0.000820,0.163315,0.000484


### Rosenbrock

In [5]:
sizes = [10, 100, 1000, 10000]
methods = ["cls", "cls-p", "thd", "thd-p", "exact", "rel", "root", "lrn", "ste"]
load_results(problem="rb", sizes=sizes, methods=methods, penalty=20)

,Method,Metric,20x4,200x4,2000x4,20000x4
0,AS,Obj Mean,63.402197,4.029966e+02,3.595975e+03,3.043025e+04
1,AS,Obj Median,61.957197,3.724574e+02,3.380251e+03,2.674693e+04
2,AS,% Infeasible,6.000000,0.000000e+00,4.000000e+00,4.100000e+01
3,AS,% Unsolved,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
4,AS,Time (Sec),0.000025,2.682686e-05,3.118992e-05,8.130789e-05
5,AS-P,Obj Mean,64.748065,4.029966e+02,3.637818e+03,inf
6,AS-P,Obj Median,62.258814,3.724574e+02,3.380251e+03,3.200610e+04
7,AS-P,% Infeasible,0.000000,0.000000e+00,0.000000e+00,5.000000e+00
8,AS-P,% Unsolved,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
9,AS-P,Time (Sec),0.000286,1.855779e-04,4.292130e-04,4.402836e-02
